In [1]:
from PureDefense import PureDefense

import os
import argparse

try: 
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.xla_multiprocessing as xmp
except: pass

from utils.utils import *

from utils.utils_purify import get_poisons, ImageListDataset, save_poisons, process_args, get_ngt

os.environ['XRT_TPU_CONFIG'] = 'localservice;0;localhost:51011'

In [2]:
### Main Function ###
def main(rank, args):

    if rank == 0:
        args.diff_steps = args.diff_steps
    elif rank == 1:
        args.diff_steps = [0,50,1]
    elif rank == 2:
        args.diff_steps = [0,10,1]
    elif rank == 3:
        args.diff_steps = [0,5,1]
    elif rank == 4:
        args.diff_steps = [0,1,1]
    elif rank == 5:
        args.diff_steps = [0,10,2]
    elif rank == 6:
        args.diff_steps = [0,50,10]
    elif rank == 7:
        args.diff_steps = [0,25,5]

    # Set the device and seed (if not None)
    device = get_device(args.device_type)
    set_seed(args.seed, device, args.device_type)

    # Process the arguments for each rank
    # args = process_args(args,rank)

    # Get the data loader and number of target indices
    if args.poison_type in [None,'NGT']:
        target_indices = 1
        purify_pbar = True
    else:
        if args.poison_type == 'Narcissus':
            target_indices = 10
        elif args.poison_type == 'GradientMatching':
            target_indices = 100
        elif args.poison_type == 'BullseyePolytope':
            if args.num_images_bp == 50:
                target_indices = 48 # 48 images in the dataset from original paper
            else:
                target_indices = 50
        elif args.poison_type == 'BullseyePolytope_Bench':
            target_indices = 100

        purify_pbar = False

    # Get diff and ebm model paths
    if args.ebm_model is not None: ebm_path = os.path.join(args.data_dir,'PureGen_Models',args.ebm_model,args.ebm_name+'.pt')
    else: ebm_path = None
    if args.diff_model is not None: diff_path = os.path.join(args.data_dir,'PureGen_Models',args.diff_model,args.diff_name+'.pt')
    else: diff_path = None

    # Create the PureDefense object
    PurifyClass = PureDefense(device,args.device_type,
                            ebm_type=args.ebm_model,ebm_path=ebm_path,ebm_nf=args.ebm_nf,
                            diff_type=args.diff_model,diff_path=diff_path,diff_nf=args.diff_nf, diff_mode=args.diff_mode,
                            verbose=args.verbose)
        
    if purify_pbar is False and rank == 0:
        pbar = tqdm(total=target_indices, desc='Purifying Poisoned Data')
    
    for i,args.target_index in enumerate(range(target_indices)):
        # Save the time that each target index takes to purify.
        if args.save_time:
            start_time = time.time()

        ### Get Data to Purify ###
        if args.poison_type is None:
            if args.dataset == 'cifar10':
                train_data = torchvision.datasets.CIFAR10(root=args.data_dir, train=True, download=(not os.path.exists(os.path.join(args.data_dir, 'cifar-10-batches-py'))), transform=torchvision.transforms.ToTensor())
                train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=False, num_workers=4)
            elif args.dataset == 'stl10':
                train_data = torchvision.datasets.STL10(root=args.data_dir, split='train', download=(not os.path.exists(os.path.join(args.data_dir, 'stl10_binary'))), transform=torchvision.transforms.ToTensor())
                train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=False, num_workers=4)
            elif args.dataset == 'stl10_64':
                train_data = torchvision.datasets.STL10(root=args.data_dir, split='train', download=(not os.path.exists(os.path.join(args.data_dir, 'stl10_binary'))), transform=torchvision.transforms.Compose([torchvision.transforms.Resize(64),torchvision.transforms.ToTensor()]))
                train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=False, num_workers=4)
            elif args.dataset == 'tinyimagenet':
                train_data = torchvision.datasets.ImageFolder(os.path.join(args.data_dir, 'tiny-imagenet-200/train'), transform=torchvision.transforms.ToTensor())
                train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=False, num_workers=4)
        elif args.poison_type == 'NGT':
            train_data = get_ngt(os.path.join(args.data_dir,'NGT'),True, transform=torchvision.transforms.ToTensor(),jpeg=args.jpeg)
            train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=False, num_workers=4)
        else:
            poison_tuple_list, poison_indices, target_mask_label = get_poisons(args,args.target_index)
            train_loader = torch.utils.data.DataLoader(ImageListDataset(poison_tuple_list), batch_size=128, shuffle=False, num_workers=4)

        if args.verbose:
            print(f'Purifying Data: {args.dataset} - {args.poison_type} - {args.target_index} of size {len(train_loader.dataset)}')

        ### Purify the dataset ###
        purified_data = PurifyClass.purify(train_loader,
                                        ebm_lang_steps=args.ebm_lang_steps,ebm_lang_temp=args.ebm_lang_temp,
                                        diff_steps=args.diff_steps, ebm_guided=args.diff_ebm_guided,
                                        sample_freq=args.diff_ebm_guided_freq, additonal_guided_steps=args.diff_ebm_additional_steps,
                                        purify_reps=args.purify_reps, pbar=purify_pbar)

        ### Save the purified data ###
        data_key = ''
        if args.ebm_lang_steps > 0 and args.ebm_model is not None:
            data_key += f'{args.ebm_model}[{args.ebm_name}]_Steps[{args.ebm_lang_steps}]_T[{args.ebm_lang_temp}]'
        if args.diff_model is not None:
            data_key += f'_{args.diff_model}[{args.diff_name}]_Steps[{args.diff_steps}]'
        if args.purify_reps > 1:
            data_key += f'_reps{args.purify_reps}'

        # if leading underscore, remove
        if data_key[0] == '_': data_key = data_key[1:]
        
        if data_key == '':
            data_key = 'Baseline'

        if args.jpeg is not None:
            data_key += f'_compressed{args.jpeg}'

        if args.poison_type is None:
            if not os.path.exists(os.path.join(args.data_dir,'PureGen_PoisonDefense',args.dataset)):
                os.makedirs(os.path.join(args.data_dir,'PureGen_PoisonDefense',args.dataset))
            torch.save(purified_data,os.path.join(args.data_dir,'PureGen_PoisonDefense',args.dataset,f'{data_key}.pt'))
        elif args.poison_type == 'NGT':
            if not os.path.exists(os.path.join(args.data_dir,'PureGen_PoisonDefense','NGT')):
                os.makedirs(os.path.join(args.data_dir,'PureGen_PoisonDefense','NGT'))
            torch.save(purified_data,os.path.join(args.data_dir,'PureGen_PoisonDefense','NGT',f'{data_key}.pt'))
        else:
            save_dir = save_poisons(args,purified_data, poison_indices, target_mask_label, data_key)

        if purify_pbar is False and rank == 0:
            # Update and set description
            pbar.update(1)
            pbar.set_description(f'Poisoned Data Saved: {data_key}')

    if purify_pbar is False and rank == 0:
        pbar.close()

    # Renendezvous
    xm.rendezvous('training end!')

In [3]:
parser = argparse.ArgumentParser(description='PyTorch Poison Attack')

### Setup Arguments ###
parser.add_argument('--remote_user', type=str, help='username for the remote server (TPU only, else pass in full directory args below)')
parser.add_argument('--num_proc', type=int, default=1, help='number of processes for TPU')
parser.add_argument('--device_type', default='xla', type=str, choices=['xla','cuda','cpu','mps'],help='device type to use')
parser.add_argument('--seed', default=11, type=int,help='seed for reproducibility')
parser.add_argument('--verbose','--v', default=False, action='store_true',help='print out additional information when running')
parser.add_argument('--data_dir', default='/home/data/', type=str, help='path to the data directory')
parser.add_argument('--jpeg', default=None, type=int, help='jpeg compression quality')

### Experiment Arguments ###
parser.add_argument('--dataset', default='cifar10', type=str, choices=['cifar10','cinic10','stl10','tinyimagenet'],help='dataset to use')
parser.add_argument('--save_time', default=True, action='store_true', help='save the time taken for the experiment')
### Purification Arguments ###

parser.add_argument('--purify_reps', default=1, type=int, help='number of purification repetitions (when using both EBM and Diffusion)')

# EBM Arguments 
args_ebm = parser.add_argument_group('EBM')
args_ebm.add_argument('--ebm_model', default='EBM', type=none_or_str, choices=[None,'SuperLightEBM','LightEBM','EBM','EBMSNGAN32','EBMSNGAN128','EBMSNGAN256'],help='type of EBM model to use')
args_ebm.add_argument('--ebm_name', default='cinic10_imagenet_ep120_nf32', type=str_or_str_list, help='path to the EBM model including train dataset')
args_ebm.add_argument('--ebm_nf', default=32, type=int_or_int_list,  help='number of filters for the ebm model')
args_ebm.add_argument('--ebm_lang_steps', default=2000, type=int_or_int_list, help='number of langevin steps')
args_ebm.add_argument('--ebm_lang_temp', default=1e-4, type=float_or_float_list, help='langevin temperature')

# Diffusion Arguments
args_diff = parser.add_argument_group('Diffusion')
args_diff.add_argument('--diff_model', default='DM_UNET_SMALL', type=none_or_str, choices=[None,'DM_UNET_SMALL', 'DM_UNET'],help='type of diffusion model to use')
args_diff.add_argument('--diff_name', default='cinic10_imagenet_nf64_modeX0_mcmc100_ep80', type=str_or_str_list, help='path to the diffusion model')
args_diff.add_argument('--diff_nf', default=64, type=int,  help='number of filters for the unet model')
args_diff.add_argument('--diff_ebm_guided', default=False, action='store_true', help='use ebm guided diffusion')
args_diff.add_argument('--diff_mode', type=str, default='X0', choices=['X0', 'Eps'], help='type of output for diffusion model')
args_diff.add_argument('--diff_steps', default=[0,50,5], nargs='+', type=int,  help='diffusion steps for sampling, [end, start, step]')
args_diff.add_argument('--diff_ebm_guided_freq', default=10, type=int, help='frequency of ebm guided diffusion steps')
args_diff.add_argument('--diff_ebm_additional_steps', default=None, type=int, help='additional ebm guided diffusion steps')

### Poison Arguments ###
parser.add_argument('--poison_type', default=None, type=str, choices=['Narcissus', 'GradientMatching','BullseyePolytope','BullseyePolytope_Bench','NGT'],help='type of poison to generate')
parser.add_argument('--poison_mode', default='from_scratch', type=str, choices=['from_scratch','transfer'],help='mode of attack')
parser.add_argument('--noise_sz_narcissus', default=32, type=int, help='size of the noise trigger for Narcissus')
parser.add_argument('--noise_eps_narcissus', default=8, type=int, help='epsilon for the noise trigger for Narcissus')
parser.add_argument('--num_images_narcissus', default=500, type=int_or_int_list, help='number of poisoned images generated')
parser.add_argument('--random_imgs_narcissus', default=False, action='store_true', help='use random images for narcissus')
parser.add_argument('--iters_bp', default=800, type=int,help='iterations for making poison')
parser.add_argument('--num_images_bp', default=50, type=int,help='number of poisoned images generated')
parser.add_argument('--net_repeat_bp', default=1, type=int, help='number of times to repeat the network for methods BP-1, BP-3, BP-5')
parser.add_argument('--num_per_class_bp', default=50, type=int, help='num of samples per class for re-training, or the poison dataset')

# Diff Models
# args = parser.parse_args(['--remote_user','sunaybhat','--num_proc','8'])
args = parser.parse_args(['--remote_user','sunaybhat','--poison_type','Narcissus','--noise_eps_narcissus','16','--num_proc','8'])

# EBM Baseline
# args = parser.parse_args(['--remote_user','sunaybhat','--diff_model','None','--num_proc','1'])
# args = parser.parse_args(['--remote_user','sunaybhat','--diff_model','None','--num_proc','1','--poison_type','Narcissus','--noise_eps_narcissus','16'])

start_time = time.time()

# Setup directories for remote server
if args.remote_user is not None:
    args.data_dir = args.data_dir.replace('/home',f'/home/{args.remote_user}')

xmp.spawn(main, args=(args,), nprocs=args.num_proc, join=True, start_method='fork')

print(f"Total time taken: {time.time() - start_time}")

Loaded DM_UNET_SMALL from /home/sunaybhat/data/PureGen_Models/DM_UNET_SMALL/cinic10_imagenet_nf64_modeX0_mcmc100_ep80.pt with 4815171 parameters


Purifying Poisoned Data:   0%|          | 0/10 [00:00<?, ?it/s]

Loaded DM_UNET_SMALL from /home/sunaybhat/data/PureGen_Models/DM_UNET_SMALL/cinic10_imagenet_nf64_modeX0_mcmc100_ep80.pt with 4815171 parameters
Loaded DM_UNET_SMALL from /home/sunaybhat/data/PureGen_Models/DM_UNET_SMALL/cinic10_imagenet_nf64_modeX0_mcmc100_ep80.pt with 4815171 parameters
Loaded DM_UNET_SMALL from /home/sunaybhat/data/PureGen_Models/DM_UNET_SMALL/cinic10_imagenet_nf64_modeX0_mcmc100_ep80.pt with 4815171 parameters
Loaded DM_UNET_SMALL from /home/sunaybhat/data/PureGen_Models/DM_UNET_SMALL/cinic10_imagenet_nf64_modeX0_mcmc100_ep80.pt with 4815171 parameters
Loaded DM_UNET_SMALL from /home/sunaybhat/data/PureGen_Models/DM_UNET_SMALL/cinic10_imagenet_nf64_modeX0_mcmc100_ep80.pt with 4815171 parameters
Loaded DM_UNET_SMALL from /home/sunaybhat/data/PureGen_Models/DM_UNET_SMALL/cinic10_imagenet_nf64_modeX0_mcmc100_ep80.pt with 4815171 parameters
Loaded DM_UNET_SMALL from /home/sunaybhat/data/PureGen_Models/DM_UNET_SMALL/cinic10_imagenet_nf64_modeX0_mcmc100_ep80.pt with 4815

Poisoned Data Saved: EBM[cinic10_imagenet_ep120_nf32]_Steps[2000]_T[0.0001]_DM_UNET_SMALL[cinic10_imagenet_nf64_modeX0_mcmc100_ep80]_Steps[[0, 50, 5]]: 100%|██████████| 10/10 [05:38<00:00, 33.81s/it]


Total time taken: 519.9320287704468


: 